# Calibrated parameters across fields and calibration runs

Compares the posterior point estimates (median or KDE mode, as chosen by
`01_calibration`) of every model parameter across fields, one box per
calibration run, using each run's final stage (`cal_yearround` or
`cal_veg`). The grey band is the range of the parameter's initial values
across fields (for soil parameters these come from SoilGrids/Saxton-Rawls,
so they vary per field).

In seasonal runs, `cal_veg` carries the soil and backscatter parameters
frozen from `cal_bare_2` -- they appear here with their stage-2 values.

Optionally, set `options.group_by` to a shapefile attribute (e.g. `Crop`) to
also compare parameters between groups of fields, with `options.group_map`
mapping its values to coarser classes (e.g. crop -> rooting-depth class).

*Author: Martina Natali (martinanatali@cnr.it, GitHub: martina01natali). License: GPL-3.0. AI assistance: developed with the help of Claude Code (Anthropic, Claude Opus 5.5) under the author's direction; see [`AI_DISCLOSURE.md`](../AI_DISCLOSURE.md).*

In [ ]:
import os

import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from wcm_swb import analysis
from wcm_swb.model import PAR_STR

file_settings = 'configuration_02_analysis_TEMPLATE.json'
options, paths = analysis.load_analysis_config(file_settings)

outputs = analysis.find_outputs(paths['root_output'], options['runs'], options['opt_obs'])
if outputs.empty:
    raise FileNotFoundError(f"No calibration outputs under {paths['root_output']} -- run 01_calibration first.")
finals = analysis.final_stage_outputs(outputs)

runs = sorted(finals.run.unique())
label = {r: options['run_labels'].get(r, r) for r in runs}

folder_analysis = paths['folder_analysis']
if paths['opt_save_plots']:
    os.makedirs(folder_analysis, exist_ok=True)

params = analysis.read_posteriors(finals)
params['run_label'] = params.run.map(label)

The likelihood's noise term `sigma` [dB] is not part of the posterior
parameter dict; take its posterior mean from each trace.

In [ ]:
sigma_rows = []
for row in finals.itertuples():
    if row.trace is None:
        continue
    posterior = az.extract(az.from_netcdf(row.trace), group='posterior', combined=True)
    if 'sigma' in posterior:
        sigma_rows.append({'run': row.run, 'field': row.field, 'stage': row.stage, 'param': 'sigma',
                           'value': float(posterior['sigma'].mean()), 'std': float(posterior['sigma'].std()),
                           'init': np.nan, 'range_lo': np.nan, 'range_hi': np.nan, 'ref': 'mean',
                           'calibrated': True, 'run_label': label[row.run]})
params = pd.concat([params, pd.DataFrame(sigma_rows)], ignore_index=True)

params.pivot_table(index='param', columns='run_label', values='value', aggfunc='median').round(3)

In [ ]:
param_order = [p for p in (*PAR_STR, 'Kc0', 'sigma') if p in set(params.param)]
titles = {
    'A': ('A', '[-]'), 'B': ('B', '[-]'), 'C': ('C', '[dB]'), 'D': ('D', r'[dB m$^3$m$^{-3}$]'),
    'Ksat': (r'K$_{sat}$', '[mm/h]'), 'lam': (r'$\lambda$', '[-]'),
    'WW_fc': (r'$\theta_{fc}$', r'[m$^3$m$^{-3}$]'), 'WW_w': (r'$\theta_{w}$', r'[m$^3$m$^{-3}$]'),
    'WW_start': (r'$\theta_{start}$', r'[m$^3$m$^{-3}$]'), 'irri_thr': (r'irri$_{thr}$', '[-]'),
    'irri_cf': (r'irri$_{cf}$', '[-]'), 'Kc0': (r'K$_{c0}$', '[-]'), 'sigma': (r'$\delta$', '[dB]'),
}


def plot_param_grid(df, x, order, filename=None):
    """One box+strip panel per parameter, grouped along `x`."""
    ncols = 3
    nrows = int(np.ceil(len(param_order) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(19 / 2.54, nrows * 6.5 / 2.54), squeeze=False)
    axes = axes.flatten()
    for ax, param in zip(axes, param_order):
        d = df[df.param == param]
        sns.boxplot(data=d, x=x, y='value', hue=x, order=order, hue_order=order, ax=ax,
                    palette='deep', legend=False, width=0.6)
        sns.stripplot(data=d, x=x, y='value', order=order, ax=ax, color='k', size=3, jitter=.1, alpha=0.5)
        if d.init.notna().any():
            if np.isclose(d.init.min(), d.init.max()):
                ax.axhline(d.init.min(), color='tab:gray', alpha=0.8, zorder=-100, lw=2)
            else:
                ax.axhspan(d.init.min(), d.init.max(), color='tab:gray', alpha=0.4, zorder=-100, lw=0)
        name, unit = titles.get(param, (param, ''))
        ax.set_ylabel(f'{name} {unit}')
        ax.set_xlabel('')
        ax.tick_params(axis='x', rotation=15)
    for ax in axes[len(param_order):]:
        fig.delaxes(ax)
    fig.subplots_adjust(wspace=0.5, hspace=0.4)
    if filename and paths['opt_save_plots']:
        fig.savefig(os.path.join(folder_analysis, filename + paths['extension_plot']), dpi=300, bbox_inches='tight')
    return fig


plot_param_grid(params, 'run_label', [label[r] for r in runs], filename='params_boxplots_runs')
plt.show()

## Parameters by field group (optional)

Runs only when `options.group_by` names an attribute present in the field
shapefiles.

In [ ]:
group_by = options.get('group_by')
if group_by:
    shapes = analysis.field_areas(paths['file_shapes'])
    if group_by not in shapes:
        raise KeyError(f'Shapefile attribute {group_by!r} not found; available: {list(shapes.columns)}')
    groups = shapes[group_by]
    if options.get('group_map'):
        groups = groups.map(options['group_map']).fillna(groups)
    params['group'] = params.field.map(groups)
    group_order = sorted(params.group.dropna().unique(), key=analysis.natural_key)
    print(params.drop_duplicates(['run', 'field']).groupby(['run_label', 'group']).field.count())
    for run in runs:
        plot_param_grid(params[params.run == run], 'group', group_order,
                        filename=f'params_boxplots_{run}_by_{group_by}')
        plt.suptitle(label[run])
        plt.show()
else:
    print('options.group_by is not set -- skipping per-group comparison.')

## Table

All point estimates, one row per field and run.

In [ ]:
table = params.pivot_table(index=['run_label', 'field'], columns='param', values='value')[param_order]
if paths['opt_save_plots']:
    table.to_csv(os.path.join(folder_analysis, 'params_per_field.csv'))
table.round(3)